# ton_iot — GraphSMOTE — All Architectures Benchmark

Merged multi-seed benchmark notebook: runs GAT, GCN, GGNN, and GraphSAGE sequentially, each with 2-seed evaluation. Produces one consolidated results JSON.

# Section 1 — Install Dependencies

In [1]:
import sys, subprocess, re

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q", *args])

subprocess.call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch-geometric", "pyg-lib", "torch-scatter",
    "torch-sparse", "torch-cluster", "torch-spline-conv"
])

import torch

torch_ver = re.match(r"(\d+\.\d+\.\d+)", torch.__version__).group(1)
cuda_ver  = torch.version.cuda

if cuda_ver is None:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
else:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver.replace('.','')}.html"

pip_install(
    "torch-geometric", "pyg-lib", "torch-scatter",
    "torch-sparse", "torch-cluster", "torch-spline-conv",
    "-f", wheel_url
)
print("Installed PyG from:", wheel_url)


Installed PyG from: https://data.pyg.org/whl/torch-2.11.0+cu128.html


## Shared Setup — Imports, Seed, Device

In [2]:
import os, time, warnings
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GATConv, GCNConv, GatedGraphConv, SAGEConv

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


Device: cuda


## Shared Setup — Configuration

In [3]:
# -- CHANGE THIS -------------------------------------------------------------
GRAPH_PT_PATH = "/content/drive/MyDrive/toniot_edge_graph.pt"
# ----------------------------------------------------------------------------

SEED = 42  # base seed; multi-seed sections below override per-run


## Shared Setup — Load Graph

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

data = torch.load(GRAPH_PT_PATH, map_location='cpu', weights_only=False)
print(data)

print(f"\nNodes         : {data.num_nodes:,}")
print(f"Edges (total) : {data.edge_index.shape[1]:,}")
print(f"Node feat dim : {data.x.shape[1]}")
print(f"Edge feat dim : {data.edge_attr.shape[1]}")

NUM_CLASSES   = int(data.edge_y.max().item()) + 1
NODE_FEAT_DIM = int(data.x.shape[1])
EDGE_FEAT_DIM = int(data.edge_attr.shape[1])

CLASS_NAMES = [
    "backdoor","ddos","dos","injection","mitm",
    "normal","password","ransomware","scanning","xss"
]

# Normalise node features on TRAIN split only (no leakage)
train_node_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_src = data.edge_index[0][data.edge_train_mask]
train_dst = data.edge_index[1][data.edge_train_mask]
train_node_mask[train_src] = True
train_node_mask[train_dst] = True

x_train = data.x[train_node_mask]
mean = x_train.mean(dim=0)
std  = x_train.std(dim=0) + 1e-8
data.x = (data.x - mean) / std

# Normalise edge features on TRAIN edges only (no leakage)
ea = data.edge_attr.clone()
for col in [0, 1, 2, 5, 6, 7]:
    m = ea[data.edge_train_mask, col].mean()
    s = ea[data.edge_train_mask, col].std() + 1e-8
    ea[:, col] = (ea[:, col] - m) / s
data.edge_attr = ea

train_edge_idx = data.edge_train_mask.nonzero(as_tuple=False).view(-1).long()
val_edge_idx   = data.edge_val_mask.nonzero(as_tuple=False).view(-1).long()
test_edge_idx  = data.edge_test_mask.nonzero(as_tuple=False).view(-1).long()

def describe_split(name, edge_idx):
    labels = data.edge_y[edge_idx]
    counts = torch.bincount(labels, minlength=NUM_CLASSES)
    total  = int(edge_idx.numel())
    txt    = ", ".join([f"{CLASS_NAMES[i]}={int(counts[i])}" for i in range(NUM_CLASSES)])
    print(f"{name:<6} edges: {total:>10,} | {txt}")

describe_split("Train", train_edge_idx)
describe_split("Val",   val_edge_idx)
describe_split("Test",  test_edge_idx)

# Fallback (src, dst) -> edge-id lookup for edge-attr resolver
edge_pair_to_id = {}
for eid, (s, d) in enumerate(data.edge_index.t().tolist()):
    edge_pair_to_id.setdefault((int(s), int(d)), int(eid))
print(f"Built edge pair lookup for {len(edge_pair_to_id):,} unique directed pairs.")


Mounted at /content/drive
Data(x=[2262, 8], edge_index=[2, 181052], edge_attr=[181052, 8], y=[2262], edge_y=[181052], edge_train_mask=[181052], edge_val_mask=[181052], edge_test_mask=[181052])

Nodes         : 2,262
Edges (total) : 181,052
Node feat dim : 8
Edge feat dim : 8
Train  edges:    126,736 | backdoor=14000, ddos=14000, dos=14000, injection=14000, mitm=736, normal=14000, password=14000, ransomware=14000, scanning=14000, xss=14000
Val    edges:     27,157 | backdoor=3000, ddos=3000, dos=3000, injection=3000, mitm=157, normal=3000, password=3000, ransomware=3000, scanning=3000, xss=3000
Test   edges:     27,159 | backdoor=3000, ddos=3000, dos=3000, injection=3000, mitm=159, normal=3000, password=3000, ransomware=3000, scanning=3000, xss=3000
Built edge pair lookup for 3,013 unique directed pairs.


## Result Collector

In [5]:
all_results = {}  # populated by each architecture section below
all_training_logs = {}  # populated by each architecture section below


---
# Architecture: GAT

## GAT — Hyperparameters

In [6]:
# ── GAT Hyperparameters ──────────────────────────────────────────────────────
# num_heads=8, hidden_dim=64 → each head works in 64-dim space.
# Node encoder projects to hidden_dim * num_heads = 512 before convolution,
# consistent with the BoT-IoT GAT notebook.
HP = dict(
    hidden_dim      = 64,
    num_layers      = 3,
    num_heads       = 8,
    dropout         = 0.3,
    lr              = 5e-4,
    weight_decay    = 1e-4,
    batch_size      = 2048,
    fan_out         = [10, 10, 10],
    max_epochs      = 100,
    patience        = 15,
    min_delta       = 5e-5,
    val_ratio       = 0.10,
    label_smoothing = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'num_heads': 8, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


## GAT — Train/Val/Test Split

# Section 4 — Loss Function

In [7]:
train_labels = data.edge_y[train_edge_idx].long()
counts = torch.bincount(train_labels, minlength=NUM_CLASSES).float().clamp(min=1)

print("Class counts (train):")
for i in range(NUM_CLASSES):
    print(f"  [{i}] {CLASS_NAMES[i]:<14} count={int(counts[i]):>8,}")

# Plain cross-entropy — no class weights.
# Class imbalance is handled by GraphSMOTE oversampling (Section 4b).
criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"\nLoss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")
print("Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).")


Class counts (train):
  [0] backdoor       count=  14,000
  [1] ddos           count=  14,000
  [2] dos            count=  14,000
  [3] injection      count=  14,000
  [4] mitm           count=     736
  [5] normal         count=  14,000
  [6] password       count=  14,000
  [7] ransomware     count=  14,000
  [8] scanning       count=  14,000
  [9] xss            count=  14,000

Loss: CrossEntropyLoss(label_smoothing=0.03)
Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).


# Section 4b — GraphSMOTE Oversampling (Training Edges Only)

GraphSMOTE (Zhao et al., 2021) addresses class imbalance by generating synthetic
minority-class embeddings in **feature space** rather than duplicating raw samples.
Applied exclusively to the **training split** — no leakage into validation or test sets.

**Algorithm (edge-level adaptation)**:
1. Build a per-edge feature vector: `concat(x[src], x[dst], edge_attr)`.
2. For each minority class, fit a *k*-NN (k=5) over the training edges of that class.
3. Interpolate between each sample and a random neighbour:
   `x_syn = x_i + λ·(x_j − x_i)`, λ ~ Uniform(0, 1).
4. Upsample until every class matches the majority-class count.
5. Synthetic feature vectors are stored in `smote_feats`/`smote_labels` and consumed
   by a dedicated DataLoader inside `train_epoch` — bypassing the GNN encoder and
   feeding directly into the shared edge MLP decoder.

In [8]:
# ── GraphSMOTE: edge-level oversampling on training split only ───────────────
# Reference: Zhao et al., "GraphSMOTE: Imbalanced Node Classification on Graphs
# with Graph Neural Networks", WSDM 2021.
#
# The edge MLP decoder expects: concat(h_src, h_dst, edge_attr)
# where h is the GNN hidden embedding (dim = hidden_dim).
# For synthetic edges we project raw node features into that same hidden_dim
# space via a small frozen linear (smote_node_proj), so synthetic vectors
# are dimensionally compatible with model.edge_mlp at training time.

import torch
from torch import Tensor

# ── Projection layer: raw node feat → hidden space ───────────────────────────
# hidden_dim is the GNN output dimension used by the edge MLP decoder.
# We tie this to HP['hidden_dim'] so it stays consistent across runs.
_smote_hidden = HP['hidden_dim']
smote_node_proj = nn.Linear(NODE_FEAT_DIM, _smote_hidden, bias=False).to('cpu')
nn.init.xavier_uniform_(smote_node_proj.weight)
smote_node_proj.eval()  # no grad needed; used only to build smote_feats

def graphsmote_edges(
    x: Tensor,           # [N, NODE_FEAT_DIM] — normalised node features
    edge_index: Tensor,  # [2, E]
    edge_attr: Tensor,   # [E, EDGE_FEAT_DIM]
    edge_idx: Tensor,    # [E_train] — indices into edge_index for training split
    edge_y: Tensor,      # [E] — full graph edge labels
    proj: nn.Module,     # Linear(NODE_FEAT_DIM -> hidden_dim)
    k: int = 5,
    seed: int = 42,
) -> tuple:
    """Return (augmented_edge_features, augmented_labels).

    augmented_edge_features : [E_aug, hidden_dim*2 + EDGE_FEAT_DIM]
    augmented_labels        : [E_aug]
    E_aug = majority_class_count * NUM_CLASSES
    """
    rng = torch.Generator(); rng.manual_seed(seed)

    src = edge_index[0, edge_idx]   # [E_train]
    dst = edge_index[1, edge_idx]   # [E_train]
    ea  = edge_attr[edge_idx]       # [E_train, EDGE_FEAT_DIM]
    lab = edge_y[edge_idx].long()   # [E_train]

    # Project raw node features into hidden_dim space (no grad)
    with torch.no_grad():
        h_src = proj(x[src])   # [E_train, hidden_dim]
        h_dst = proj(x[dst])   # [E_train, hidden_dim]

    # Per-edge feature vector: [h_src || h_dst || edge_attr]
    # Matches the exact input shape of model.edge_mlp
    feats = torch.cat([h_src, h_dst, ea], dim=1)   # [E_train, hidden_dim*2+EDGE_FEAT_DIM]

    num_classes = int(lab.max().item()) + 1
    counts      = torch.bincount(lab, minlength=num_classes)
    majority_n  = int(counts.max().item())

    aug_feats, aug_labs = [feats], [lab]

    for cls in range(num_classes):
        mask   = (lab == cls).nonzero(as_tuple=False).view(-1)
        n_cls  = mask.numel()
        n_need = majority_n - n_cls
        if n_need <= 0:
            continue   # already at majority size

        cls_feats = feats[mask]   # [n_cls, D]
        k_eff     = min(k, n_cls - 1) if n_cls > 1 else 1

        if k_eff == 0:
            noise = torch.randn(n_need, cls_feats.shape[1], generator=rng) * 1e-6
            synth = cls_feats.expand(n_need, -1) + noise
        else:
            diff   = cls_feats.unsqueeze(1) - cls_feats.unsqueeze(0)   # [n, n, D]
            dists  = (diff ** 2).sum(-1)                                # [n, n]
            dists.fill_diagonal_(float('inf'))
            nn_idx = dists.topk(k_eff, largest=False, dim=1).indices   # [n, k]

            anchor_ids      = torch.randint(0, n_cls,  (n_need,), generator=rng)
            nn_slot_ids     = torch.randint(0, k_eff,  (n_need,), generator=rng)
            lam             = torch.rand(n_need, 1,               generator=rng)
            anchors         = cls_feats[anchor_ids]
            neighbour_local = nn_idx[anchor_ids, nn_slot_ids]
            neighbours      = cls_feats[neighbour_local]
            synth           = anchors + lam * (neighbours - anchors)

        aug_feats.append(synth)
        aug_labs.append(torch.full((n_need,), cls, dtype=torch.long))

    aug_feats = torch.cat(aug_feats, dim=0)
    aug_labs  = torch.cat(aug_labs,  dim=0)

    perm = torch.randperm(aug_feats.shape[0], generator=rng)
    return aug_feats[perm], aug_labs[perm]


print("Running GraphSMOTE on training edges …")
smote_feats, smote_labels = graphsmote_edges(
    x          = data.x,
    edge_index  = data.edge_index,
    edge_attr   = data.edge_attr,
    edge_idx    = train_edge_idx,
    edge_y      = data.edge_y,
    proj        = smote_node_proj,
    k           = 5,
    seed        = SEED,
)
expected_dim = _smote_hidden * 2 + EDGE_FEAT_DIM
print(f"  smote_feats dim : {smote_feats.shape[1]}  (expected {expected_dim} = hidden_dim*2 + EDGE_FEAT_DIM)")
assert smote_feats.shape[1] == expected_dim,     f"Dimension mismatch: smote_feats={smote_feats.shape[1]}, edge_mlp expects={expected_dim}"
print(f"  Original train edges : {train_edge_idx.numel():,}")
print(f"  Augmented train edges: {smote_feats.shape[0]:,}")
new_counts = torch.bincount(smote_labels, minlength=NUM_CLASSES)
print("  Class distribution after GraphSMOTE:")
for i in range(NUM_CLASSES):
    print(f"    [{i}] {CLASS_NAMES[i]:<14} {int(new_counts[i]):>8,}")

# ── SMOTE DataLoader ─────────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader

smote_dataset = TensorDataset(smote_feats, smote_labels)
smote_loader  = TorchDataLoader(
    smote_dataset,
    batch_size=HP['batch_size'],
    shuffle=True,
    drop_last=False,
)
print(f"  SMOTE DataLoader: {len(smote_loader)} batches  (feat dim matches edge_mlp ✓)")


Running GraphSMOTE on training edges …
  smote_feats dim : 136  (expected 136 = hidden_dim*2 + EDGE_FEAT_DIM)
  Original train edges : 126,736
  Augmented train edges: 140,000
  Class distribution after GraphSMOTE:
    [0] backdoor         14,000
    [1] ddos             14,000
    [2] dos              14,000
    [3] injection        14,000
    [4] mitm             14,000
    [5] normal           14,000
    [6] password         14,000
    [7] ransomware       14,000
    [8] scanning         14,000
    [9] xss              14,000
  SMOTE DataLoader: 69 batches  (feat dim matches edge_mlp ✓)


# Section 5 — DataLoaders (structural leakage fix)

In [9]:
# ── Structural-leakage fix (matches BoT-IoT benchmark) ──────────────────────
# Train/val loaders use a train-only MP graph so test edges never appear
# in neighbourhood sampling during training.
# Test loader uses the full graph (training complete, no gradient flow).

def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )
    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_y[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )



def resolve_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    if hasattr(batch, 'input_id') and batch.input_id is not None:
        global_ids = seed_edge_idx.cpu()[batch.input_id.cpu().long()]
        return full_edge_attr[global_ids].to(batch.x.device)
    edge_label_index = batch.edge_label_index
    if hasattr(batch, 'n_id') and int(edge_label_index.max()) < int(batch.n_id.numel()):
        src = batch.n_id[edge_label_index[0].cpu().long()]
        dst = batch.n_id[edge_label_index[1].cpu().long()]
    else:
        src = edge_label_index[0].cpu().long()
        dst = edge_label_index[1].cpu().long()
    edge_ids = torch.tensor(
        [pair_lookup[(int(s), int(d))] for s, d in zip(src.tolist(), dst.tolist())],
        dtype=torch.long
    )
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Edge-attribute resolver ready.


# Section 6 — GAT (GATConv v1) Model Definition

In [10]:
class GATEdgeClassifier(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim, num_heads,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout      = dropout
        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim * num_heads)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(num_layers):
            is_last = (i == num_layers - 1)
            concat  = not is_last
            self.convs.append(GATConv(
                hidden_dim * num_heads, hidden_dim,
                heads=num_heads, concat=concat, dropout=dropout,
            ))
            out_dim = hidden_dim * num_heads if concat else hidden_dim
            self.norms.append(nn.LayerNorm(out_dim))
        final_node_dim = hidden_dim
        mlp_in = final_node_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.elu(self.node_encoder(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index); h = norm(h); h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        return self.edge_mlp(torch.cat([h[src], h[dst], edge_attr], dim=-1))



# Section 8 — Train / Eval Functions

In [11]:
def train_epoch():
    model.train()
    total_loss, total_acc, n = 0., 0., 0

    # ── (1) Real graph mini-batches: full GNN pipeline ───────────────────────
    for batch in train_loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, train_edge_idx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs

    # ── (2) GraphSMOTE batches: synthetic edges → edge MLP decoder only ──────
    # Synthetic feature vectors = [h_src || h_dst || edge_attr] (interpolated).
    # Graph convolution is skipped; they feed directly into model.edge_mlp.
    for sf, sy in smote_loader:
        sf, sy  = sf.to(DEVICE), sy.to(DEVICE)
        logits  = model.edge_mlp(sf)
        loss    = criterion(logits, sy)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = sy.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == sy).float().sum().item()
        n          += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_split(loader):
    model.eval()
    if loader is train_loader:
        eidx = train_edge_idx
    elif loader is val_loader:
        eidx = val_edge_idx
    else:
        eidx = test_edge_idx
    total_loss, total_acc, n = 0., 0., 0
    all_preds, all_labels = [], []
    for batch in loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, eidx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs
        all_preds.append(logits.argmax(-1).cpu())
        all_labels.append(y.cpu())
    f1 = f1_score(torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy(),
                  average='macro', zero_division=0)
    return total_acc / n, total_loss / n, f1


# Section 9 — Training Loop

## Section — Multi-Seed Evaluation (added)

In [12]:
# ============================================================
# Section 16 — Multi-Seed Evaluation (GAT, ton_iot, GraphSMOTE)
# ============================================================
# NOTE: train_epoch()/evaluate_split() close over GLOBAL `model`, `optimizer`,
# `criterion`, `train_loader`, `val_loader`, `test_loader`, `smote_loader`.
# This cell reassigns those globals each seed iteration.
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GATEdgeClassifier
MODEL_TAG    = "GAT"
DATASET_TAG  = "ton_iot"
STRATEGY_TAG = "GraphSMOTE"

SEEDS = [0, 1]

def build_smote(train_edge_idx_r, seed):
    proj = nn.Linear(NODE_FEAT_DIM, HP['hidden_dim'], bias=False).to('cpu')
    nn.init.xavier_uniform_(proj.weight)
    proj.eval()
    sf, sy = graphsmote_edges(
        x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr,
        edge_idx=train_edge_idx_r, edge_y=data.edge_y, proj=proj, k=5, seed=seed,
    )
    ds = TensorDataset(sf, sy)
    return TorchDataLoader(ds, batch_size=HP['batch_size'], shuffle=True, drop_last=False)

def run_once(seed):
    global model, optimizer, scheduler, criterion, train_loader, val_loader, test_loader, smote_loader
    set_seed(seed)

    train_edge_idx_r = train_edge_idx
    val_edge_idx_r   = val_edge_idx

    criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

    smote_loader = build_smote(train_edge_idx_r, seed)

    train_loader = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_heads     = HP['num_heads'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state, best_epoch = -float('inf'), 0, None, 0
    history = []
    for epoch in range(1, HP['max_epochs'] + 1):
        tr_loss, tr_acc = train_epoch()
        val_acc, val_loss, val_f1 = evaluate_split(val_loader)
        scheduler.step(val_f1)
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1, best_epoch, patience_count = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model.load_state_dict(best_state)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            ea = resolve_edge_attr(batch, test_edge_idx, data.edge_attr, edge_pair_to_id)
            logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
            all_preds.append(logits.argmax(-1).cpu())
            all_labels.append(batch.edge_label.cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    test_acc       = accuracy_score(y_true, y_pred)
    test_macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        CLASS_NAMES[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=best_epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
        history=history,
    )

results = [run_once(s) for s in SEEDS]

training_logs = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS,
    per_seed_history={r['seed']: r.pop('history') for r in results},
)

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in CLASS_NAMES:
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

all_results["GAT"] = summary
all_training_logs["GAT"] = training_logs



  GAT — ton_iot / GraphSMOTE — 3-seed summary
  acc       : 0.9293 ± 0.0006
  macro_f1  : 0.9076 ± 0.0015
  precision : 0.9343 ± 0.0006
  recall    : 0.8965 ± 0.0021

  Per-class F1 (mean ± std):
    backdoor        : 0.9988 ± 0.0002
    ddos            : 0.9301 ± 0.0067
    dos             : 0.9812 ± 0.0004
    injection       : 0.8195 ± 0.0018
    mitm            : 0.7191 ± 0.0135
    normal          : 0.9990 ± 0.0006
    password        : 0.7656 ± 0.0014
    ransomware      : 0.9937 ± 0.0003
    scanning        : 0.9632 ± 0.0023
    xss             : 0.9054 ± 0.0023


---
# Architecture: GCN

## GCN — Hyperparameters

In [13]:
# ── GCN Hyperparameters ──────────────────────────────────────────────────────
HP = dict(
    hidden_dim      = 64,
    num_layers      = 3,
    dropout         = 0.3,
    lr              = 5e-4,
    weight_decay    = 1e-4,
    batch_size      = 2048,
    fan_out         = [10, 10, 10],
    max_epochs      = 100,
    patience        = 15,
    min_delta       = 5e-5,
    val_ratio       = 0.10,
    label_smoothing = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


## GCN — Train/Val/Test Split

# Section 4 — Loss Function

In [14]:
train_labels = data.edge_y[train_edge_idx].long()
counts = torch.bincount(train_labels, minlength=NUM_CLASSES).float().clamp(min=1)

print("Class counts (train):")
for i in range(NUM_CLASSES):
    print(f"  [{i}] {CLASS_NAMES[i]:<14} count={int(counts[i]):>8,}")

# Plain cross-entropy — no class weights.
# Class imbalance is handled by GraphSMOTE oversampling (Section 4b).
criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"\nLoss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")
print("Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).")


Class counts (train):
  [0] backdoor       count=  14,000
  [1] ddos           count=  14,000
  [2] dos            count=  14,000
  [3] injection      count=  14,000
  [4] mitm           count=     736
  [5] normal         count=  14,000
  [6] password       count=  14,000
  [7] ransomware     count=  14,000
  [8] scanning       count=  14,000
  [9] xss            count=  14,000

Loss: CrossEntropyLoss(label_smoothing=0.03)
Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).


# Section 4b — GraphSMOTE Oversampling (Training Edges Only)

GraphSMOTE (Zhao et al., 2021) addresses class imbalance by generating synthetic
minority-class embeddings in **feature space** rather than duplicating raw samples.
Applied exclusively to the **training split** — no leakage into validation or test sets.

**Algorithm (edge-level adaptation)**:
1. Build a per-edge feature vector: `concat(x[src], x[dst], edge_attr)`.
2. For each minority class, fit a *k*-NN (k=5) over the training edges of that class.
3. Interpolate between each sample and a random neighbour:
   `x_syn = x_i + λ·(x_j − x_i)`, λ ~ Uniform(0, 1).
4. Upsample until every class matches the majority-class count.
5. Synthetic feature vectors are stored in `smote_feats`/`smote_labels` and consumed
   by a dedicated DataLoader inside `train_epoch` — bypassing the GNN encoder and
   feeding directly into the shared edge MLP decoder.

In [15]:
# ── GraphSMOTE: edge-level oversampling on training split only ───────────────
# Reference: Zhao et al., "GraphSMOTE: Imbalanced Node Classification on Graphs
# with Graph Neural Networks", WSDM 2021.
#
# The edge MLP decoder expects: concat(h_src, h_dst, edge_attr)
# where h is the GNN hidden embedding (dim = hidden_dim).
# For synthetic edges we project raw node features into that same hidden_dim
# space via a small frozen linear (smote_node_proj), so synthetic vectors
# are dimensionally compatible with model.edge_mlp at training time.

import torch
from torch import Tensor

# ── Projection layer: raw node feat → hidden space ───────────────────────────
# hidden_dim is the GNN output dimension used by the edge MLP decoder.
# We tie this to HP['hidden_dim'] so it stays consistent across runs.
_smote_hidden = HP['hidden_dim']
smote_node_proj = nn.Linear(NODE_FEAT_DIM, _smote_hidden, bias=False).to('cpu')
nn.init.xavier_uniform_(smote_node_proj.weight)
smote_node_proj.eval()  # no grad needed; used only to build smote_feats

def graphsmote_edges(
    x: Tensor,           # [N, NODE_FEAT_DIM] — normalised node features
    edge_index: Tensor,  # [2, E]
    edge_attr: Tensor,   # [E, EDGE_FEAT_DIM]
    edge_idx: Tensor,    # [E_train] — indices into edge_index for training split
    edge_y: Tensor,      # [E] — full graph edge labels
    proj: nn.Module,     # Linear(NODE_FEAT_DIM -> hidden_dim)
    k: int = 5,
    seed: int = 42,
) -> tuple:
    """Return (augmented_edge_features, augmented_labels).

    augmented_edge_features : [E_aug, hidden_dim*2 + EDGE_FEAT_DIM]
    augmented_labels        : [E_aug]
    E_aug = majority_class_count * NUM_CLASSES
    """
    rng = torch.Generator(); rng.manual_seed(seed)

    src = edge_index[0, edge_idx]   # [E_train]
    dst = edge_index[1, edge_idx]   # [E_train]
    ea  = edge_attr[edge_idx]       # [E_train, EDGE_FEAT_DIM]
    lab = edge_y[edge_idx].long()   # [E_train]

    # Project raw node features into hidden_dim space (no grad)
    with torch.no_grad():
        h_src = proj(x[src])   # [E_train, hidden_dim]
        h_dst = proj(x[dst])   # [E_train, hidden_dim]

    # Per-edge feature vector: [h_src || h_dst || edge_attr]
    # Matches the exact input shape of model.edge_mlp
    feats = torch.cat([h_src, h_dst, ea], dim=1)   # [E_train, hidden_dim*2+EDGE_FEAT_DIM]

    num_classes = int(lab.max().item()) + 1
    counts      = torch.bincount(lab, minlength=num_classes)
    majority_n  = int(counts.max().item())

    aug_feats, aug_labs = [feats], [lab]

    for cls in range(num_classes):
        mask   = (lab == cls).nonzero(as_tuple=False).view(-1)
        n_cls  = mask.numel()
        n_need = majority_n - n_cls
        if n_need <= 0:
            continue   # already at majority size

        cls_feats = feats[mask]   # [n_cls, D]
        k_eff     = min(k, n_cls - 1) if n_cls > 1 else 1

        if k_eff == 0:
            noise = torch.randn(n_need, cls_feats.shape[1], generator=rng) * 1e-6
            synth = cls_feats.expand(n_need, -1) + noise
        else:
            diff   = cls_feats.unsqueeze(1) - cls_feats.unsqueeze(0)   # [n, n, D]
            dists  = (diff ** 2).sum(-1)                                # [n, n]
            dists.fill_diagonal_(float('inf'))
            nn_idx = dists.topk(k_eff, largest=False, dim=1).indices   # [n, k]

            anchor_ids      = torch.randint(0, n_cls,  (n_need,), generator=rng)
            nn_slot_ids     = torch.randint(0, k_eff,  (n_need,), generator=rng)
            lam             = torch.rand(n_need, 1,               generator=rng)
            anchors         = cls_feats[anchor_ids]
            neighbour_local = nn_idx[anchor_ids, nn_slot_ids]
            neighbours      = cls_feats[neighbour_local]
            synth           = anchors + lam * (neighbours - anchors)

        aug_feats.append(synth)
        aug_labs.append(torch.full((n_need,), cls, dtype=torch.long))

    aug_feats = torch.cat(aug_feats, dim=0)
    aug_labs  = torch.cat(aug_labs,  dim=0)

    perm = torch.randperm(aug_feats.shape[0], generator=rng)
    return aug_feats[perm], aug_labs[perm]


print("Running GraphSMOTE on training edges …")
smote_feats, smote_labels = graphsmote_edges(
    x          = data.x,
    edge_index  = data.edge_index,
    edge_attr   = data.edge_attr,
    edge_idx    = train_edge_idx,
    edge_y      = data.edge_y,
    proj        = smote_node_proj,
    k           = 5,
    seed        = SEED,
)
expected_dim = _smote_hidden * 2 + EDGE_FEAT_DIM
print(f"  smote_feats dim : {smote_feats.shape[1]}  (expected {expected_dim} = hidden_dim*2 + EDGE_FEAT_DIM)")
assert smote_feats.shape[1] == expected_dim,     f"Dimension mismatch: smote_feats={smote_feats.shape[1]}, edge_mlp expects={expected_dim}"
print(f"  Original train edges : {train_edge_idx.numel():,}")
print(f"  Augmented train edges: {smote_feats.shape[0]:,}")
new_counts = torch.bincount(smote_labels, minlength=NUM_CLASSES)
print("  Class distribution after GraphSMOTE:")
for i in range(NUM_CLASSES):
    print(f"    [{i}] {CLASS_NAMES[i]:<14} {int(new_counts[i]):>8,}")

# ── SMOTE DataLoader ─────────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader

smote_dataset = TensorDataset(smote_feats, smote_labels)
smote_loader  = TorchDataLoader(
    smote_dataset,
    batch_size=HP['batch_size'],
    shuffle=True,
    drop_last=False,
)
print(f"  SMOTE DataLoader: {len(smote_loader)} batches  (feat dim matches edge_mlp ✓)")


Running GraphSMOTE on training edges …
  smote_feats dim : 136  (expected 136 = hidden_dim*2 + EDGE_FEAT_DIM)
  Original train edges : 126,736
  Augmented train edges: 140,000
  Class distribution after GraphSMOTE:
    [0] backdoor         14,000
    [1] ddos             14,000
    [2] dos              14,000
    [3] injection        14,000
    [4] mitm             14,000
    [5] normal           14,000
    [6] password         14,000
    [7] ransomware       14,000
    [8] scanning         14,000
    [9] xss              14,000
  SMOTE DataLoader: 69 batches  (feat dim matches edge_mlp ✓)


# Section 5 — DataLoaders (structural leakage fix)

In [16]:
# ── Structural-leakage fix (matches BoT-IoT benchmark) ──────────────────────
# Train/val loaders use a train-only MP graph so test edges never appear
# in neighbourhood sampling during training.
# Test loader uses the full graph (training complete, no gradient flow).

def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )
    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_y[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )



def resolve_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    if hasattr(batch, 'input_id') and batch.input_id is not None:
        global_ids = seed_edge_idx.cpu()[batch.input_id.cpu().long()]
        return full_edge_attr[global_ids].to(batch.x.device)
    edge_label_index = batch.edge_label_index
    if hasattr(batch, 'n_id') and int(edge_label_index.max()) < int(batch.n_id.numel()):
        src = batch.n_id[edge_label_index[0].cpu().long()]
        dst = batch.n_id[edge_label_index[1].cpu().long()]
    else:
        src = edge_label_index[0].cpu().long()
        dst = edge_label_index[1].cpu().long()
    edge_ids = torch.tensor(
        [pair_lookup[(int(s), int(d))] for s, d in zip(src.tolist(), dst.tolist())],
        dtype=torch.long
    )
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Edge-attribute resolver ready.


# Section 6 — GCN Model Definition

In [17]:
class GCNEdgeClassifier(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout      = dropout
        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim, normalize=True))
            self.norms.append(nn.LayerNorm(hidden_dim))
        mlp_in = hidden_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index); h = norm(h); h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        return self.edge_mlp(torch.cat([h[src], h[dst], edge_attr], dim=-1))



# Section 8 — Train / Eval Functions

In [18]:
def train_epoch():
    model.train()
    total_loss, total_acc, n = 0., 0., 0

    # ── (1) Real graph mini-batches: full GNN pipeline ───────────────────────
    for batch in train_loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, train_edge_idx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs

    # ── (2) GraphSMOTE batches: synthetic edges → edge MLP decoder only ──────
    # Synthetic feature vectors = [h_src || h_dst || edge_attr] (interpolated).
    # Graph convolution is skipped; they feed directly into model.edge_mlp.
    for sf, sy in smote_loader:
        sf, sy  = sf.to(DEVICE), sy.to(DEVICE)
        logits  = model.edge_mlp(sf)
        loss    = criterion(logits, sy)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = sy.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == sy).float().sum().item()
        n          += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_split(loader):
    model.eval()
    if loader is train_loader:
        eidx = train_edge_idx
    elif loader is val_loader:
        eidx = val_edge_idx
    else:
        eidx = test_edge_idx
    total_loss, total_acc, n = 0., 0., 0
    all_preds, all_labels = [], []
    for batch in loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, eidx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs
        all_preds.append(logits.argmax(-1).cpu())
        all_labels.append(y.cpu())
    f1 = f1_score(torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy(),
                  average='macro', zero_division=0)
    return total_acc / n, total_loss / n, f1


# Section 9 — Training Loop

## Section — Multi-Seed Evaluation (added)

In [19]:
# ============================================================
# Section 16 — Multi-Seed Evaluation (GCN, ton_iot, GraphSMOTE)
# ============================================================
# NOTE: train_epoch()/evaluate_split() close over GLOBAL `model`, `optimizer`,
# `criterion`, `train_loader`, `val_loader`, `test_loader`, `smote_loader`.
# This cell reassigns those globals each seed iteration.
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GCNEdgeClassifier
MODEL_TAG    = "GCN"
DATASET_TAG  = "ton_iot"
STRATEGY_TAG = "GraphSMOTE"

SEEDS = [0, 1]

def build_smote(train_edge_idx_r, seed):
    proj = nn.Linear(NODE_FEAT_DIM, HP['hidden_dim'], bias=False).to('cpu')
    nn.init.xavier_uniform_(proj.weight)
    proj.eval()
    sf, sy = graphsmote_edges(
        x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr,
        edge_idx=train_edge_idx_r, edge_y=data.edge_y, proj=proj, k=5, seed=seed,
    )
    ds = TensorDataset(sf, sy)
    return TorchDataLoader(ds, batch_size=HP['batch_size'], shuffle=True, drop_last=False)

def run_once(seed):
    global model, optimizer, scheduler, criterion, train_loader, val_loader, test_loader, smote_loader
    set_seed(seed)

    train_edge_idx_r = train_edge_idx
    val_edge_idx_r   = val_edge_idx

    criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

    smote_loader = build_smote(train_edge_idx_r, seed)

    train_loader = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state, best_epoch = -float('inf'), 0, None, 0
    history = []
    for epoch in range(1, HP['max_epochs'] + 1):
        tr_loss, tr_acc = train_epoch()
        val_acc, val_loss, val_f1 = evaluate_split(val_loader)
        scheduler.step(val_f1)
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1, best_epoch, patience_count = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model.load_state_dict(best_state)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            ea = resolve_edge_attr(batch, test_edge_idx, data.edge_attr, edge_pair_to_id)
            logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
            all_preds.append(logits.argmax(-1).cpu())
            all_labels.append(batch.edge_label.cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    test_acc       = accuracy_score(y_true, y_pred)
    test_macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        CLASS_NAMES[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=best_epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
        history=history,
    )

results = [run_once(s) for s in SEEDS]

training_logs = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS,
    per_seed_history={r['seed']: r.pop('history') for r in results},
)

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in CLASS_NAMES:
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

all_results["GCN"] = summary
all_training_logs["GCN"] = training_logs



  GCN — ton_iot / GraphSMOTE — 3-seed summary
  acc       : 0.9187 ± 0.0058
  macro_f1  : 0.8902 ± 0.0149
  precision : 0.9221 ± 0.0059
  recall    : 0.8775 ± 0.0160

  Per-class F1 (mean ± std):
    backdoor        : 0.9987 ± 0.0002
    ddos            : 0.9003 ± 0.0190
    dos             : 0.9791 ± 0.0037
    injection       : 0.7955 ± 0.0111
    mitm            : 0.6338 ± 0.1162
    normal          : 0.9956 ± 0.0011
    password        : 0.7558 ± 0.0139
    ransomware      : 0.9933 ± 0.0006
    scanning        : 0.9573 ± 0.0046
    xss             : 0.8924 ± 0.0004


---
# Architecture: GGNN

## GGNN — Hyperparameters

In [20]:
# ── GGNN Hyperparameters ─────────────────────────────────────────────────────
# NOTE: num_layers = GRU unroll steps inside GatedGraphConv (not stacked layers).
# num_layers=3 gives 3-hop coverage with shared weights, same as GCN/SAGE/GAT.
HP = dict(
    hidden_dim      = 64,
    num_layers      = 3,
    dropout         = 0.3,
    lr              = 5e-4,
    weight_decay    = 1e-4,
    batch_size      = 2048,
    fan_out         = [10, 10, 10],
    max_epochs      = 100,
    patience        = 15,
    min_delta       = 5e-5,
    val_ratio       = 0.10,
    label_smoothing = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


## GGNN — Train/Val/Test Split

# Section 4 — Loss Function

In [21]:
train_labels = data.edge_y[train_edge_idx].long()
counts = torch.bincount(train_labels, minlength=NUM_CLASSES).float().clamp(min=1)

print("Class counts (train):")
for i in range(NUM_CLASSES):
    print(f"  [{i}] {CLASS_NAMES[i]:<14} count={int(counts[i]):>8,}")

# Plain cross-entropy — no class weights.
# Class imbalance is handled by GraphSMOTE oversampling (Section 4b).
criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"\nLoss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")
print("Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).")


Class counts (train):
  [0] backdoor       count=  14,000
  [1] ddos           count=  14,000
  [2] dos            count=  14,000
  [3] injection      count=  14,000
  [4] mitm           count=     736
  [5] normal         count=  14,000
  [6] password       count=  14,000
  [7] ransomware     count=  14,000
  [8] scanning       count=  14,000
  [9] xss            count=  14,000

Loss: CrossEntropyLoss(label_smoothing=0.03)
Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).


# Section 4b — GraphSMOTE Oversampling (Training Edges Only)

GraphSMOTE (Zhao et al., 2021) addresses class imbalance by generating synthetic
minority-class embeddings in **feature space** rather than duplicating raw samples.
Applied exclusively to the **training split** — no leakage into validation or test sets.

**Algorithm (edge-level adaptation)**:
1. Build a per-edge feature vector: `concat(x[src], x[dst], edge_attr)`.
2. For each minority class, fit a *k*-NN (k=5) over the training edges of that class.
3. Interpolate between each sample and a random neighbour:
   `x_syn = x_i + λ·(x_j − x_i)`, λ ~ Uniform(0, 1).
4. Upsample until every class matches the majority-class count.
5. Synthetic feature vectors are stored in `smote_feats`/`smote_labels` and consumed
   by a dedicated DataLoader inside `train_epoch` — bypassing the GNN encoder and
   feeding directly into the shared edge MLP decoder.

In [22]:
# ── GraphSMOTE: edge-level oversampling on training split only ───────────────
# Reference: Zhao et al., "GraphSMOTE: Imbalanced Node Classification on Graphs
# with Graph Neural Networks", WSDM 2021.
#
# The edge MLP decoder expects: concat(h_src, h_dst, edge_attr)
# where h is the GNN hidden embedding (dim = hidden_dim).
# For synthetic edges we project raw node features into that same hidden_dim
# space via a small frozen linear (smote_node_proj), so synthetic vectors
# are dimensionally compatible with model.edge_mlp at training time.

import torch
from torch import Tensor

# ── Projection layer: raw node feat → hidden space ───────────────────────────
# hidden_dim is the GNN output dimension used by the edge MLP decoder.
# We tie this to HP['hidden_dim'] so it stays consistent across runs.
_smote_hidden = HP['hidden_dim']
smote_node_proj = nn.Linear(NODE_FEAT_DIM, _smote_hidden, bias=False).to('cpu')
nn.init.xavier_uniform_(smote_node_proj.weight)
smote_node_proj.eval()  # no grad needed; used only to build smote_feats

def graphsmote_edges(
    x: Tensor,           # [N, NODE_FEAT_DIM] — normalised node features
    edge_index: Tensor,  # [2, E]
    edge_attr: Tensor,   # [E, EDGE_FEAT_DIM]
    edge_idx: Tensor,    # [E_train] — indices into edge_index for training split
    edge_y: Tensor,      # [E] — full graph edge labels
    proj: nn.Module,     # Linear(NODE_FEAT_DIM -> hidden_dim)
    k: int = 5,
    seed: int = 42,
) -> tuple:
    """Return (augmented_edge_features, augmented_labels).

    augmented_edge_features : [E_aug, hidden_dim*2 + EDGE_FEAT_DIM]
    augmented_labels        : [E_aug]
    E_aug = majority_class_count * NUM_CLASSES
    """
    rng = torch.Generator(); rng.manual_seed(seed)

    src = edge_index[0, edge_idx]   # [E_train]
    dst = edge_index[1, edge_idx]   # [E_train]
    ea  = edge_attr[edge_idx]       # [E_train, EDGE_FEAT_DIM]
    lab = edge_y[edge_idx].long()   # [E_train]

    # Project raw node features into hidden_dim space (no grad)
    with torch.no_grad():
        h_src = proj(x[src])   # [E_train, hidden_dim]
        h_dst = proj(x[dst])   # [E_train, hidden_dim]

    # Per-edge feature vector: [h_src || h_dst || edge_attr]
    # Matches the exact input shape of model.edge_mlp
    feats = torch.cat([h_src, h_dst, ea], dim=1)   # [E_train, hidden_dim*2+EDGE_FEAT_DIM]

    num_classes = int(lab.max().item()) + 1
    counts      = torch.bincount(lab, minlength=num_classes)
    majority_n  = int(counts.max().item())

    aug_feats, aug_labs = [feats], [lab]

    for cls in range(num_classes):
        mask   = (lab == cls).nonzero(as_tuple=False).view(-1)
        n_cls  = mask.numel()
        n_need = majority_n - n_cls
        if n_need <= 0:
            continue   # already at majority size

        cls_feats = feats[mask]   # [n_cls, D]
        k_eff     = min(k, n_cls - 1) if n_cls > 1 else 1

        if k_eff == 0:
            noise = torch.randn(n_need, cls_feats.shape[1], generator=rng) * 1e-6
            synth = cls_feats.expand(n_need, -1) + noise
        else:
            diff   = cls_feats.unsqueeze(1) - cls_feats.unsqueeze(0)   # [n, n, D]
            dists  = (diff ** 2).sum(-1)                                # [n, n]
            dists.fill_diagonal_(float('inf'))
            nn_idx = dists.topk(k_eff, largest=False, dim=1).indices   # [n, k]

            anchor_ids      = torch.randint(0, n_cls,  (n_need,), generator=rng)
            nn_slot_ids     = torch.randint(0, k_eff,  (n_need,), generator=rng)
            lam             = torch.rand(n_need, 1,               generator=rng)
            anchors         = cls_feats[anchor_ids]
            neighbour_local = nn_idx[anchor_ids, nn_slot_ids]
            neighbours      = cls_feats[neighbour_local]
            synth           = anchors + lam * (neighbours - anchors)

        aug_feats.append(synth)
        aug_labs.append(torch.full((n_need,), cls, dtype=torch.long))

    aug_feats = torch.cat(aug_feats, dim=0)
    aug_labs  = torch.cat(aug_labs,  dim=0)

    perm = torch.randperm(aug_feats.shape[0], generator=rng)
    return aug_feats[perm], aug_labs[perm]


print("Running GraphSMOTE on training edges …")
smote_feats, smote_labels = graphsmote_edges(
    x          = data.x,
    edge_index  = data.edge_index,
    edge_attr   = data.edge_attr,
    edge_idx    = train_edge_idx,
    edge_y      = data.edge_y,
    proj        = smote_node_proj,
    k           = 5,
    seed        = SEED,
)
expected_dim = _smote_hidden * 2 + EDGE_FEAT_DIM
print(f"  smote_feats dim : {smote_feats.shape[1]}  (expected {expected_dim} = hidden_dim*2 + EDGE_FEAT_DIM)")
assert smote_feats.shape[1] == expected_dim,     f"Dimension mismatch: smote_feats={smote_feats.shape[1]}, edge_mlp expects={expected_dim}"
print(f"  Original train edges : {train_edge_idx.numel():,}")
print(f"  Augmented train edges: {smote_feats.shape[0]:,}")
new_counts = torch.bincount(smote_labels, minlength=NUM_CLASSES)
print("  Class distribution after GraphSMOTE:")
for i in range(NUM_CLASSES):
    print(f"    [{i}] {CLASS_NAMES[i]:<14} {int(new_counts[i]):>8,}")

# ── SMOTE DataLoader ─────────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader

smote_dataset = TensorDataset(smote_feats, smote_labels)
smote_loader  = TorchDataLoader(
    smote_dataset,
    batch_size=HP['batch_size'],
    shuffle=True,
    drop_last=False,
)
print(f"  SMOTE DataLoader: {len(smote_loader)} batches  (feat dim matches edge_mlp ✓)")


Running GraphSMOTE on training edges …
  smote_feats dim : 136  (expected 136 = hidden_dim*2 + EDGE_FEAT_DIM)
  Original train edges : 126,736
  Augmented train edges: 140,000
  Class distribution after GraphSMOTE:
    [0] backdoor         14,000
    [1] ddos             14,000
    [2] dos              14,000
    [3] injection        14,000
    [4] mitm             14,000
    [5] normal           14,000
    [6] password         14,000
    [7] ransomware       14,000
    [8] scanning         14,000
    [9] xss              14,000
  SMOTE DataLoader: 69 batches  (feat dim matches edge_mlp ✓)


# Section 5 — DataLoaders (structural leakage fix)

In [23]:
# ── Structural-leakage fix (matches BoT-IoT benchmark) ──────────────────────
# Train/val loaders use a train-only MP graph so test edges never appear
# in neighbourhood sampling during training.
# Test loader uses the full graph (training complete, no gradient flow).

def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )
    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_y[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )



def resolve_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    if hasattr(batch, 'input_id') and batch.input_id is not None:
        global_ids = seed_edge_idx.cpu()[batch.input_id.cpu().long()]
        return full_edge_attr[global_ids].to(batch.x.device)
    edge_label_index = batch.edge_label_index
    if hasattr(batch, 'n_id') and int(edge_label_index.max()) < int(batch.n_id.numel()):
        src = batch.n_id[edge_label_index[0].cpu().long()]
        dst = batch.n_id[edge_label_index[1].cpu().long()]
    else:
        src = edge_label_index[0].cpu().long()
        dst = edge_label_index[1].cpu().long()
    edge_ids = torch.tensor(
        [pair_lookup[(int(s), int(d))] for s, d in zip(src.tolist(), dst.tolist())],
        dtype=torch.long
    )
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Edge-attribute resolver ready.


# Section 6 — GGNN Model Definition

In [24]:
class GGNNEdgeClassifier(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout      = dropout
        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim)
        self.ggnn         = GatedGraphConv(hidden_dim, num_layers)
        self.norm         = nn.LayerNorm(hidden_dim)
        mlp_in = hidden_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        h = self.ggnn(h, edge_index)
        h = self.norm(h); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        return self.edge_mlp(torch.cat([h[src], h[dst], edge_attr], dim=-1))



# Section 8 — Train / Eval Functions

In [25]:
def train_epoch():
    model.train()
    total_loss, total_acc, n = 0., 0., 0

    # ── (1) Real graph mini-batches: full GNN pipeline ───────────────────────
    for batch in train_loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, train_edge_idx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs

    # ── (2) GraphSMOTE batches: synthetic edges → edge MLP decoder only ──────
    # Synthetic feature vectors = [h_src || h_dst || edge_attr] (interpolated).
    # Graph convolution is skipped; they feed directly into model.edge_mlp.
    for sf, sy in smote_loader:
        sf, sy  = sf.to(DEVICE), sy.to(DEVICE)
        logits  = model.edge_mlp(sf)
        loss    = criterion(logits, sy)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = sy.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == sy).float().sum().item()
        n          += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_split(loader):
    model.eval()
    if loader is train_loader:
        eidx = train_edge_idx
    elif loader is val_loader:
        eidx = val_edge_idx
    else:
        eidx = test_edge_idx
    total_loss, total_acc, n = 0., 0., 0
    all_preds, all_labels = [], []
    for batch in loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, eidx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs
        all_preds.append(logits.argmax(-1).cpu())
        all_labels.append(y.cpu())
    f1 = f1_score(torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy(),
                  average='macro', zero_division=0)
    return total_acc / n, total_loss / n, f1


# Section 9 — Training Loop

## Section — Multi-Seed Evaluation (added)

In [26]:
# ============================================================
# Section 16 — Multi-Seed Evaluation (GGNN, ton_iot, GraphSMOTE)
# ============================================================
# NOTE: train_epoch()/evaluate_split() close over GLOBAL `model`, `optimizer`,
# `criterion`, `train_loader`, `val_loader`, `test_loader`, `smote_loader`.
# This cell reassigns those globals each seed iteration.
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GGNNEdgeClassifier
MODEL_TAG    = "GGNN"
DATASET_TAG  = "ton_iot"
STRATEGY_TAG = "GraphSMOTE"

SEEDS = [0, 1]

def build_smote(train_edge_idx_r, seed):
    proj = nn.Linear(NODE_FEAT_DIM, HP['hidden_dim'], bias=False).to('cpu')
    nn.init.xavier_uniform_(proj.weight)
    proj.eval()
    sf, sy = graphsmote_edges(
        x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr,
        edge_idx=train_edge_idx_r, edge_y=data.edge_y, proj=proj, k=5, seed=seed,
    )
    ds = TensorDataset(sf, sy)
    return TorchDataLoader(ds, batch_size=HP['batch_size'], shuffle=True, drop_last=False)

def run_once(seed):
    global model, optimizer, scheduler, criterion, train_loader, val_loader, test_loader, smote_loader
    set_seed(seed)

    train_edge_idx_r = train_edge_idx
    val_edge_idx_r   = val_edge_idx

    criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

    smote_loader = build_smote(train_edge_idx_r, seed)

    train_loader = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state, best_epoch = -float('inf'), 0, None, 0
    history = []
    for epoch in range(1, HP['max_epochs'] + 1):
        tr_loss, tr_acc = train_epoch()
        val_acc, val_loss, val_f1 = evaluate_split(val_loader)
        scheduler.step(val_f1)
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1, best_epoch, patience_count = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model.load_state_dict(best_state)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            ea = resolve_edge_attr(batch, test_edge_idx, data.edge_attr, edge_pair_to_id)
            logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
            all_preds.append(logits.argmax(-1).cpu())
            all_labels.append(batch.edge_label.cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    test_acc       = accuracy_score(y_true, y_pred)
    test_macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        CLASS_NAMES[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=best_epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
        history=history,
    )

results = [run_once(s) for s in SEEDS]

training_logs = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS,
    per_seed_history={r['seed']: r.pop('history') for r in results},
)

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in CLASS_NAMES:
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

all_results["GGNN"] = summary
all_training_logs["GGNN"] = training_logs



  GGNN — ton_iot / GraphSMOTE — 3-seed summary
  acc       : 0.9463 ± 0.0010
  macro_f1  : 0.9273 ± 0.0010
  precision : 0.9462 ± 0.0013
  recall    : 0.9160 ± 0.0008

  Per-class F1 (mean ± std):
    backdoor        : 0.9989 ± 0.0002
    ddos            : 0.9520 ± 0.0022
    dos             : 0.9829 ± 0.0002
    injection       : 0.8519 ± 0.0018
    mitm            : 0.7513 ± 0.0038
    normal          : 0.9999 ± 0.0001
    password        : 0.8441 ± 0.0029
    ransomware      : 0.9984 ± 0.0014
    scanning        : 0.9827 ± 0.0031
    xss             : 0.9111 ± 0.0015


---
# Architecture: GraphSAGE

## GraphSAGE — Hyperparameters

In [27]:
# ── GraphSAGE Hyperparameters ────────────────────────────────────────────────
HP = dict(
    hidden_dim      = 64,
    num_layers      = 3,
    dropout         = 0.3,
    lr              = 5e-4,
    weight_decay    = 1e-4,
    batch_size      = 2048,
    fan_out         = [10, 10, 10],
    max_epochs      = 100,
    patience        = 15,
    min_delta       = 5e-5,
    val_ratio       = 0.10,
    label_smoothing = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


## GraphSAGE — Train/Val/Test Split

# Section 4 — Loss Function

In [28]:
train_labels = data.edge_y[train_edge_idx].long()
counts = torch.bincount(train_labels, minlength=NUM_CLASSES).float().clamp(min=1)

print("Class counts (train):")
for i in range(NUM_CLASSES):
    print(f"  [{i}] {CLASS_NAMES[i]:<14} count={int(counts[i]):>8,}")

# Plain cross-entropy — no class weights.
# Class imbalance is handled by GraphSMOTE oversampling (Section 4b).
criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"\nLoss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")
print("Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).")


Class counts (train):
  [0] backdoor       count=  14,000
  [1] ddos           count=  14,000
  [2] dos            count=  14,000
  [3] injection      count=  14,000
  [4] mitm           count=     736
  [5] normal         count=  14,000
  [6] password       count=  14,000
  [7] ransomware     count=  14,000
  [8] scanning       count=  14,000
  [9] xss            count=  14,000

Loss: CrossEntropyLoss(label_smoothing=0.03)
Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 4b).


# Section 4b — GraphSMOTE Oversampling (Training Edges Only)

GraphSMOTE (Zhao et al., 2021) addresses class imbalance by generating synthetic
minority-class embeddings in **feature space** rather than duplicating raw samples.
Applied exclusively to the **training split** — no leakage into validation or test sets.

**Algorithm (edge-level adaptation)**:
1. Build a per-edge feature vector: `concat(x[src], x[dst], edge_attr)`.
2. For each minority class, fit a *k*-NN (k=5) over the training edges of that class.
3. Interpolate between each sample and a random neighbour:
   `x_syn = x_i + λ·(x_j − x_i)`, λ ~ Uniform(0, 1).
4. Upsample until every class matches the majority-class count.
5. Synthetic feature vectors are stored in `smote_feats`/`smote_labels` and consumed
   by a dedicated DataLoader inside `train_epoch` — bypassing the GNN encoder and
   feeding directly into the shared edge MLP decoder.

In [29]:
# ── GraphSMOTE: edge-level oversampling on training split only ───────────────
# Reference: Zhao et al., "GraphSMOTE: Imbalanced Node Classification on Graphs
# with Graph Neural Networks", WSDM 2021.
#
# The edge MLP decoder expects: concat(h_src, h_dst, edge_attr)
# where h is the GNN hidden embedding (dim = hidden_dim).
# For synthetic edges we project raw node features into that same hidden_dim
# space via a small frozen linear (smote_node_proj), so synthetic vectors
# are dimensionally compatible with model.edge_mlp at training time.

import torch
from torch import Tensor

# ── Projection layer: raw node feat → hidden space ───────────────────────────
# hidden_dim is the GNN output dimension used by the edge MLP decoder.
# We tie this to HP['hidden_dim'] so it stays consistent across runs.
_smote_hidden = HP['hidden_dim']
smote_node_proj = nn.Linear(NODE_FEAT_DIM, _smote_hidden, bias=False).to('cpu')
nn.init.xavier_uniform_(smote_node_proj.weight)
smote_node_proj.eval()  # no grad needed; used only to build smote_feats

def graphsmote_edges(
    x: Tensor,           # [N, NODE_FEAT_DIM] — normalised node features
    edge_index: Tensor,  # [2, E]
    edge_attr: Tensor,   # [E, EDGE_FEAT_DIM]
    edge_idx: Tensor,    # [E_train] — indices into edge_index for training split
    edge_y: Tensor,      # [E] — full graph edge labels
    proj: nn.Module,     # Linear(NODE_FEAT_DIM -> hidden_dim)
    k: int = 5,
    seed: int = 42,
) -> tuple:
    """Return (augmented_edge_features, augmented_labels).

    augmented_edge_features : [E_aug, hidden_dim*2 + EDGE_FEAT_DIM]
    augmented_labels        : [E_aug]
    E_aug = majority_class_count * NUM_CLASSES
    """
    rng = torch.Generator(); rng.manual_seed(seed)

    src = edge_index[0, edge_idx]   # [E_train]
    dst = edge_index[1, edge_idx]   # [E_train]
    ea  = edge_attr[edge_idx]       # [E_train, EDGE_FEAT_DIM]
    lab = edge_y[edge_idx].long()   # [E_train]

    # Project raw node features into hidden_dim space (no grad)
    with torch.no_grad():
        h_src = proj(x[src])   # [E_train, hidden_dim]
        h_dst = proj(x[dst])   # [E_train, hidden_dim]

    # Per-edge feature vector: [h_src || h_dst || edge_attr]
    # Matches the exact input shape of model.edge_mlp
    feats = torch.cat([h_src, h_dst, ea], dim=1)   # [E_train, hidden_dim*2+EDGE_FEAT_DIM]

    num_classes = int(lab.max().item()) + 1
    counts      = torch.bincount(lab, minlength=num_classes)
    majority_n  = int(counts.max().item())

    aug_feats, aug_labs = [feats], [lab]

    for cls in range(num_classes):
        mask   = (lab == cls).nonzero(as_tuple=False).view(-1)
        n_cls  = mask.numel()
        n_need = majority_n - n_cls
        if n_need <= 0:
            continue   # already at majority size

        cls_feats = feats[mask]   # [n_cls, D]
        k_eff     = min(k, n_cls - 1) if n_cls > 1 else 1

        if k_eff == 0:
            noise = torch.randn(n_need, cls_feats.shape[1], generator=rng) * 1e-6
            synth = cls_feats.expand(n_need, -1) + noise
        else:
            diff   = cls_feats.unsqueeze(1) - cls_feats.unsqueeze(0)   # [n, n, D]
            dists  = (diff ** 2).sum(-1)                                # [n, n]
            dists.fill_diagonal_(float('inf'))
            nn_idx = dists.topk(k_eff, largest=False, dim=1).indices   # [n, k]

            anchor_ids      = torch.randint(0, n_cls,  (n_need,), generator=rng)
            nn_slot_ids     = torch.randint(0, k_eff,  (n_need,), generator=rng)
            lam             = torch.rand(n_need, 1,               generator=rng)
            anchors         = cls_feats[anchor_ids]
            neighbour_local = nn_idx[anchor_ids, nn_slot_ids]
            neighbours      = cls_feats[neighbour_local]
            synth           = anchors + lam * (neighbours - anchors)

        aug_feats.append(synth)
        aug_labs.append(torch.full((n_need,), cls, dtype=torch.long))

    aug_feats = torch.cat(aug_feats, dim=0)
    aug_labs  = torch.cat(aug_labs,  dim=0)

    perm = torch.randperm(aug_feats.shape[0], generator=rng)
    return aug_feats[perm], aug_labs[perm]


print("Running GraphSMOTE on training edges …")
smote_feats, smote_labels = graphsmote_edges(
    x          = data.x,
    edge_index  = data.edge_index,
    edge_attr   = data.edge_attr,
    edge_idx    = train_edge_idx,
    edge_y      = data.edge_y,
    proj        = smote_node_proj,
    k           = 5,
    seed        = SEED,
)
expected_dim = _smote_hidden * 2 + EDGE_FEAT_DIM
print(f"  smote_feats dim : {smote_feats.shape[1]}  (expected {expected_dim} = hidden_dim*2 + EDGE_FEAT_DIM)")
assert smote_feats.shape[1] == expected_dim,     f"Dimension mismatch: smote_feats={smote_feats.shape[1]}, edge_mlp expects={expected_dim}"
print(f"  Original train edges : {train_edge_idx.numel():,}")
print(f"  Augmented train edges: {smote_feats.shape[0]:,}")
new_counts = torch.bincount(smote_labels, minlength=NUM_CLASSES)
print("  Class distribution after GraphSMOTE:")
for i in range(NUM_CLASSES):
    print(f"    [{i}] {CLASS_NAMES[i]:<14} {int(new_counts[i]):>8,}")

# ── SMOTE DataLoader ─────────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader

smote_dataset = TensorDataset(smote_feats, smote_labels)
smote_loader  = TorchDataLoader(
    smote_dataset,
    batch_size=HP['batch_size'],
    shuffle=True,
    drop_last=False,
)
print(f"  SMOTE DataLoader: {len(smote_loader)} batches  (feat dim matches edge_mlp ✓)")


Running GraphSMOTE on training edges …
  smote_feats dim : 136  (expected 136 = hidden_dim*2 + EDGE_FEAT_DIM)
  Original train edges : 126,736
  Augmented train edges: 140,000
  Class distribution after GraphSMOTE:
    [0] backdoor         14,000
    [1] ddos             14,000
    [2] dos              14,000
    [3] injection        14,000
    [4] mitm             14,000
    [5] normal           14,000
    [6] password         14,000
    [7] ransomware       14,000
    [8] scanning         14,000
    [9] xss              14,000
  SMOTE DataLoader: 69 batches  (feat dim matches edge_mlp ✓)


# Section 5 — DataLoaders (structural leakage fix)

In [30]:
# ── Structural-leakage fix (matches BoT-IoT benchmark) ──────────────────────
# Train/val loaders use a train-only MP graph so test edges never appear
# in neighbourhood sampling during training.
# Test loader uses the full graph (training complete, no gradient flow).

def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )
    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_y[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )



def resolve_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    if hasattr(batch, 'input_id') and batch.input_id is not None:
        global_ids = seed_edge_idx.cpu()[batch.input_id.cpu().long()]
        return full_edge_attr[global_ids].to(batch.x.device)
    edge_label_index = batch.edge_label_index
    if hasattr(batch, 'n_id') and int(edge_label_index.max()) < int(batch.n_id.numel()):
        src = batch.n_id[edge_label_index[0].cpu().long()]
        dst = batch.n_id[edge_label_index[1].cpu().long()]
    else:
        src = edge_label_index[0].cpu().long()
        dst = edge_label_index[1].cpu().long()
    edge_ids = torch.tensor(
        [pair_lookup[(int(s), int(d))] for s, d in zip(src.tolist(), dst.tolist())],
        dtype=torch.long
    )
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Edge-attribute resolver ready.


# Section 6 — GraphSAGE Model Definition

In [31]:
class GraphSAGEEdgeClassifier(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout      = dropout
        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
        mlp_in = hidden_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index); h = norm(h); h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        return self.edge_mlp(torch.cat([h[src], h[dst], edge_attr], dim=-1))



# Section 8 — Train / Eval Functions

In [32]:
def train_epoch():
    model.train()
    total_loss, total_acc, n = 0., 0., 0

    # ── (1) Real graph mini-batches: full GNN pipeline ───────────────────────
    for batch in train_loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, train_edge_idx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs

    # ── (2) GraphSMOTE batches: synthetic edges → edge MLP decoder only ──────
    # Synthetic feature vectors = [h_src || h_dst || edge_attr] (interpolated).
    # Graph convolution is skipped; they feed directly into model.edge_mlp.
    for sf, sy in smote_loader:
        sf, sy  = sf.to(DEVICE), sy.to(DEVICE)
        logits  = model.edge_mlp(sf)
        loss    = criterion(logits, sy)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = sy.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == sy).float().sum().item()
        n          += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_split(loader):
    model.eval()
    if loader is train_loader:
        eidx = train_edge_idx
    elif loader is val_loader:
        eidx = val_edge_idx
    else:
        eidx = test_edge_idx
    total_loss, total_acc, n = 0., 0., 0
    all_preds, all_labels = [], []
    for batch in loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, eidx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs
        all_preds.append(logits.argmax(-1).cpu())
        all_labels.append(y.cpu())
    f1 = f1_score(torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy(),
                  average='macro', zero_division=0)
    return total_acc / n, total_loss / n, f1


# Section 9 — Training Loop

## Section — Multi-Seed Evaluation (added)

In [33]:
# ============================================================
# Section 16 — Multi-Seed Evaluation (GraphSAGE, ton_iot, GraphSMOTE)
# ============================================================
# NOTE: train_epoch()/evaluate_split() close over GLOBAL `model`, `optimizer`,
# `criterion`, `train_loader`, `val_loader`, `test_loader`, `smote_loader`.
# This cell reassigns those globals each seed iteration.
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GraphSAGEEdgeClassifier
MODEL_TAG    = "GraphSAGE"
DATASET_TAG  = "ton_iot"
STRATEGY_TAG = "GraphSMOTE"

SEEDS = [0, 1]

def build_smote(train_edge_idx_r, seed):
    proj = nn.Linear(NODE_FEAT_DIM, HP['hidden_dim'], bias=False).to('cpu')
    nn.init.xavier_uniform_(proj.weight)
    proj.eval()
    sf, sy = graphsmote_edges(
        x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr,
        edge_idx=train_edge_idx_r, edge_y=data.edge_y, proj=proj, k=5, seed=seed,
    )
    ds = TensorDataset(sf, sy)
    return TorchDataLoader(ds, batch_size=HP['batch_size'], shuffle=True, drop_last=False)

def run_once(seed):
    global model, optimizer, scheduler, criterion, train_loader, val_loader, test_loader, smote_loader
    set_seed(seed)

    train_edge_idx_r = train_edge_idx
    val_edge_idx_r   = val_edge_idx

    criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

    smote_loader = build_smote(train_edge_idx_r, seed)

    train_loader = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state, best_epoch = -float('inf'), 0, None, 0
    history = []
    for epoch in range(1, HP['max_epochs'] + 1):
        tr_loss, tr_acc = train_epoch()
        val_acc, val_loss, val_f1 = evaluate_split(val_loader)
        scheduler.step(val_f1)
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1, best_epoch, patience_count = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model.load_state_dict(best_state)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            ea = resolve_edge_attr(batch, test_edge_idx, data.edge_attr, edge_pair_to_id)
            logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
            all_preds.append(logits.argmax(-1).cpu())
            all_labels.append(batch.edge_label.cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    test_acc       = accuracy_score(y_true, y_pred)
    test_macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        CLASS_NAMES[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=best_epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
        history=history,
    )

results = [run_once(s) for s in SEEDS]

training_logs = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS,
    per_seed_history={r['seed']: r.pop('history') for r in results},
)

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in CLASS_NAMES:
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

all_results["GraphSAGE"] = summary
all_training_logs["GraphSAGE"] = training_logs



  GraphSAGE — ton_iot / GraphSMOTE — 3-seed summary
  acc       : 0.9435 ± 0.0022
  macro_f1  : 0.9238 ± 0.0023
  precision : 0.9430 ± 0.0008
  recall    : 0.9127 ± 0.0027

  Per-class F1 (mean ± std):
    backdoor        : 0.9990 ± 0.0000
    ddos            : 0.9479 ± 0.0029
    dos             : 0.9826 ± 0.0006
    injection       : 0.8480 ± 0.0038
    mitm            : 0.7421 ± 0.0063
    normal          : 0.9998 ± 0.0002
    password        : 0.8317 ± 0.0102
    ransomware      : 0.9960 ± 0.0015
    scanning        : 0.9785 ± 0.0035
    xss             : 0.9127 ± 0.0004


---
## Consolidated Results — Save All Architectures

In [34]:
import json, os

consolidated = dict(
    dataset="ton_iot",
    strategy="GraphSMOTE",
    architectures=all_results,
)

out_dir = "/content/results" if os.path.isdir("/content") else "."
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/ton_iot_GraphSMOTE_all_architectures_multiseed.json"
with open(out_path, "w") as f:
    json.dump(consolidated, f, indent=2)
print(f"Saved consolidated results: {out_path}")

consolidated_logs = dict(
    dataset="ton_iot",
    strategy="GraphSMOTE",
    architectures=all_training_logs,
)
logs_path = f"{out_dir}/ton_iot_GraphSMOTE_all_architectures_trainlogs.json"
with open(logs_path, "w") as f:
    json.dump(consolidated_logs, f, indent=2)
print(f"Saved consolidated training logs: {logs_path}")

print("\n" + "="*70)
print(f"  SUMMARY — ton_iot / GraphSMOTE")
print("="*70)
for model_tag, summary in all_results.items():
    print(f"\n{model_tag}:")
    for metric, stat in summary['overall'].items():
        print(f"    {metric:<10}: {stat['mean']:.4f} \u00b1 {stat['std']:.4f}")

try:
    from google.colab import files as colab_files
    colab_files.download(out_path)
    colab_files.download(logs_path)
except ImportError:
    pass


Saved consolidated results: /content/results/ton_iot_GraphSMOTE_all_architectures_multiseed.json

  SUMMARY — ton_iot / GraphSMOTE

GAT:
    acc       : 0.9293 ± 0.0006
    macro_f1  : 0.9076 ± 0.0015
    precision : 0.9343 ± 0.0006
    recall    : 0.8965 ± 0.0021

GCN:
    acc       : 0.9187 ± 0.0058
    macro_f1  : 0.8902 ± 0.0149
    precision : 0.9221 ± 0.0059
    recall    : 0.8775 ± 0.0160

GGNN:
    acc       : 0.9463 ± 0.0010
    macro_f1  : 0.9273 ± 0.0010
    precision : 0.9462 ± 0.0013
    recall    : 0.9160 ± 0.0008

GraphSAGE:
    acc       : 0.9435 ± 0.0022
    macro_f1  : 0.9238 ± 0.0023
    precision : 0.9430 ± 0.0008
    recall    : 0.9127 ± 0.0027


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>